In [1]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier,
    AdaBoostClassifier, VotingClassifier, StackingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, f1_score, roc_curve,
    confusion_matrix, balanced_accuracy_score, average_precision_score,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

pd.set_option('display.max_rows', 300)
pd.set_option('display.width', 200)

# 패널 생성 (등록대수 + 운영정보)

In [2]:
# 파일 경로 지정 + 존재 확인 + shape 추적 도구
DATA_DIR = Path('/Users/ichaeu/Desktop/EVen')

EV_REG_FILE      = DATA_DIR / '전국전기차등록대수.csv'
CHARGER_STD_FILE = DATA_DIR / '전기차충전소표준데이터.csv'     
CHARGER_OPS_FILE = DATA_DIR / '전기차충전소위치및운영정보.csv'
SUBSIDY_FILE     = DATA_DIR / '무공해차보조금현황.xlsx'  
HOUSING_FILE = DATA_DIR / 'scratch_housing_indicators.csv'
POP_FILE     = DATA_DIR / 'scratch_ev_full_merged.csv' 

# 파일명 오타 찾기
for f in (EV_REG_FILE, CHARGER_STD_FILE, CHARGER_OPS_FILE, SUBSIDY_FILE,
          HOUSING_FILE, POP_FILE):
    print('OK      ' if f.exists() else 'MISSING ', f.name)


# shape 추적
_shape_log = [] # (단계명, 행수, 열수)

def log_shape(df, label):
    """현재 shape과 직전 단계 대비 증감을 출력."""
    n_row, n_col = df.shape
    if _shape_log:
        prev_label, prev_row, prev_col = _shape_log[-1]
        delta = f'  (행 {n_row - prev_row:+,} / 열 {n_col - prev_col:+d}  ← {prev_label})'
    else:
        delta = '  (기준점)'
    print(f'[{label:<26}] {n_row:>8,} 행 × {n_col:>3} 열{delta}')
    _shape_log.append((label, n_row, n_col))
    return df

def reset_shape_log():
    _shape_log.clear()

OK       전국전기차등록대수.csv
OK       전기차충전소표준데이터.csv
OK       전기차충전소위치및운영정보.csv
OK       무공해차보조금현황.xlsx
OK       scratch_housing_indicators.csv
OK       scratch_ev_full_merged.csv


In [3]:
# 지역명 표기 통일 규칙 → 전부 축약형으로 통일
SIDO_FULL_TO_SHORT = {
    '서울특별시': '서울', '인천광역시': '인천', '경기도': '경기',
    '강원도': '강원', '강원특별자치도': '강원',          # 개편 전후 표기 모두 대응
    '충청북도': '충북', '충청남도': '충남', '대전광역시': '대전',
    '세종특별자치시': '세종', '경상북도': '경북', '대구광역시': '대구',
    '전라북도': '전북', '전북특별자치도': '전북', '전라남도': '전남',
    '광주광역시': '광주', '경상남도': '경남', '부산광역시': '부산',
    '울산광역시': '울산', '제주특별자치도': '제주',
}
SIDO_FULL_NAMES = set(SIDO_FULL_TO_SHORT)

# 일반구를 가진 시: '성남시 분당구'까지 살려야 등록대수 데이터와 단위가 맞는다
CITIES_WITH_GU = {'고양시', '성남시', '수원시', '안산시', '안양시', '용인시',
                  '창원시', '청주시', '천안시', '전주시', '포항시'}

In [4]:
# 주소 → (시도, 시군구) 키 추출
# (A) 충전소: 도로명 전체 주소 → parse_sido_sigungu / attach_region_keys
# (B) 등록대수: '서울 강남구' → split_sido_sigungu

def parse_sido_sigungu(tokens):
    """(A) 주소 토큰 리스트 -> (시도 풀네임, 시군구) """
    if len(tokens) < 2 or tokens[0] == '-':      # 토큰 부족 / 결측 표기
        return None, None
    sido = tokens[0]
    if sido not in SIDO_FULL_NAMES:              # 시도명으로 시작하지 않으면 형식 깨짐
        return None, None
    if sido == '세종특별자치시': # 단층제 → 하위 시군구 없음
        return sido, '세종특별자치시'
    if len(tokens) >= 3 and tokens[1] in CITIES_WITH_GU and tokens[2].endswith('구'):
        return sido, tokens[1] + ' ' + tokens[2]  # '성남시 분당구'
    return sido, tokens[1]


def attach_region_keys(raw_df, addr_col, label):
    """(A) sido_short/sigungu 컬럼 부착 + 파싱 실패 행 제외 """
    addr_parts = raw_df[addr_col].astype(str).str.split().apply(parse_sido_sigungu)
    out = raw_df.copy()
    out['sido_full'] = addr_parts.apply(lambda t: t[0])
    out['sigungu']   = addr_parts.apply(lambda t: t[1])

    parsed_df = out[out['sido_full'].notna()].copy() # 실패 행 제거
    parsed_df['sido_short'] = parsed_df['sido_full'].map(SIDO_FULL_TO_SHORT)

    # 군위군 경북 → 대구 편입
    gunwi = (parsed_df['sido_short'] == '경북') & (parsed_df['sigungu'] == '군위군')
    parsed_df.loc[gunwi, 'sido_short'] = '대구'

    n_drop = len(raw_df) - len(parsed_df)
    log_shape(parsed_df, f'{label} (실패 {n_drop:,}건 제외)')
    return parsed_df


def split_sido_sigungu(s):
    """(B) '서울 강남구' / '경상북도 포항시 남구' -> (시도 축약형, 시군구)."""
    sido, sigungu = s.split(' ', 1) # 첫 공백에서만 분리
    return SIDO_FULL_TO_SHORT.get(sido, sido), sigungu # 풀네임이면 변환, 축약형이면 통과

In [5]:
# 전기차 등록대수: 로드 → 전기만 필터 → 시군구 합계
# 원본은 차종별·용도별로 행이 쪼개져 있어 시군구당 여러 행 → 합산 필요
reset_shape_log()

ev_reg = pd.read_csv(EV_REG_FILE, encoding='cp949')
log_shape(ev_reg, '등록대수 원본')

ev_reg = ev_reg[ev_reg['연료별'] == '전기'].copy()  # 타 연료 제거
log_shape(ev_reg, '연료=전기 필터')

# merge 키 생성
key_parts = ev_reg['시군구별'].apply(split_sido_sigungu)
ev_reg['sido_short'] = key_parts.apply(lambda t: t[0])
ev_reg['sigungu']    = key_parts.apply(lambda t: t[1])
log_shape(ev_reg, '지역 키 컬럼 추가')

# 부천시 일반구 폐지 → '부천시 소사구' 등을 '부천시'로 통합
ev_reg.loc[ev_reg['sigungu'].str.startswith('부천시'), 'sigungu'] = '부천시'

# 축약형만 나와야 정상.
print('시도 목록:', sorted(ev_reg['sido_short'].unique()))

ev_reg_by_sigungu = (
    ev_reg.groupby(['sido_short', 'sigungu'])['계'].sum()
          .reset_index()
          .rename(columns={'계': 'registration'})
)
log_shape(ev_reg_by_sigungu, '등록대수 시군구 집계')
ev_reg_by_sigungu.head()

[등록대수 원본                   ]      508 행 ×   8 열  (기준점)
[연료=전기 필터                  ]      508 행 ×   8 열  (행 +0 / 열 +0  ← 등록대수 원본)
[지역 키 컬럼 추가                ]      508 행 ×  10 열  (행 +0 / 열 +2  ← 연료=전기 필터)
시도 목록: ['강원', '경기', '경남', '경북', '광주', '대구', '대전', '부산', '서울', '세종', '울산', '인천', '전남', '전북', '제주', '충남', '충북']
[등록대수 시군구 집계               ]      250 행 ×   3 열  (행 -258 / 열 -7  ← 지역 키 컬럼 추가)


,sido_short,sigungu,registration
0,강원,강릉시,3671
1,강원,고성군,364
2,강원,동해시,872
3,강원,삼척시,729
4,강원,속초시,887


In [6]:
# 충전소 운영정보: 로드 → 주소 파싱 → 플래그 → 시군구 집계
# 레코드 1행 = 충전기 1대 (충전소 단위 아님)
charger_ops = pd.read_csv(CHARGER_OPS_FILE, encoding='cp949', low_memory=False)
log_shape(charger_ops, '충전소 원본')

charger_ops = attach_region_keys(charger_ops, '주소', label='충전소 주소 파싱')

# 범주형 → 0/1 플래그 (groupby.sum()으로 개수를 세기 위함)
charger_ops['is_fast'] = (charger_ops['기종(대)'] == '급속').astype(int)
charger_ops['is_slow'] = (charger_ops['기종(대)'] == '완속').astype(int)
charger_ops['is_apt']  = (charger_ops['시설구분(대)'] == '공동주택시설').astype(int)
log_shape(charger_ops, '플래그 컬럼 추가')

charger_by_sigungu = charger_ops.groupby(['sido_short', 'sigungu']).agg(
    total_chargers    =('충전기ID', 'count'),
    fast_chargers     =('is_fast', 'sum'),
    slow_chargers     =('is_slow', 'sum'),
    apt_charger_count =('is_apt',  'sum'),
).reset_index()
log_shape(charger_by_sigungu, '충전기 시군구 집계')

charger_by_sigungu['apt_charger_ratio'] = ( # 공동주택 설치 비중
    charger_by_sigungu['apt_charger_count'] / charger_by_sigungu['total_chargers']
)
log_shape(charger_by_sigungu, '비율 파생변수 추가')
charger_by_sigungu.head()

[충전소 원본                    ]  273,281 행 ×  15 열  (행 +273,031 / 열 +12  ← 등록대수 시군구 집계)
[충전소 주소 파싱 (실패 510건 제외)    ]  272,771 행 ×  18 열  (행 -510 / 열 +3  ← 충전소 원본)
[플래그 컬럼 추가                 ]  272,771 행 ×  21 열  (행 +0 / 열 +3  ← 충전소 주소 파싱 (실패 510건 제외))
[충전기 시군구 집계                ]      266 행 ×   6 열  (행 -272,505 / 열 -15  ← 플래그 컬럼 추가)
[비율 파생변수 추가                ]      266 행 ×   7 열  (행 +0 / 열 +1  ← 충전기 시군구 집계)


,sido_short,sigungu,total_chargers,fast_chargers,slow_chargers,apt_charger_count,apt_charger_ratio
0,강원,강릉시,1473,273,1200,634,0.430414
1,강원,고성군,228,62,166,44,0.192982
2,강원,동해시,476,56,420,286,0.600840
3,강원,삼척시,507,142,365,154,0.303748
4,강원,속초시,506,83,423,210,0.415020


In [7]:
# 두 집계표를 (시도, 시군구)로 결합
panel_outer = ev_reg_by_sigungu.merge(
    charger_by_sigungu, on=['sido_short', 'sigungu'], how='outer', indicator=True
)
log_shape(panel_outer, 'outer merge')
print(panel_outer['_merge'].value_counts()) # both / left_only / right_only

sigungu_panel = (
    panel_outer[panel_outer['_merge'] == 'both'] # 양쪽 다 있는 시군구만
    .drop(columns='_merge')
    .query('registration > 0') # 등록대수 0 제외
    .copy()
)
log_shape(sigungu_panel, '최종 패널')
sigungu_panel.head()

[outer merge               ]      266 행 ×   9 열  (행 +0 / 열 +2  ← 비율 파생변수 추가)
_merge
both          250
right_only     16
left_only       0
Name: count, dtype: int64
[최종 패널                     ]      250 행 ×   8 열  (행 -16 / 열 -1  ← outer merge)


,sido_short,sigungu,registration,total_chargers,fast_chargers,slow_chargers,apt_charger_count,apt_charger_ratio
0,강원,강릉시,3671.0,1473,273,1200,634,0.430414
1,강원,고성군,364.0,228,62,166,44,0.192982
2,강원,동해시,872.0,476,56,420,286,0.600840
3,강원,삼척시,729.0,507,142,365,154,0.303748
4,강원,속초시,887.0,506,83,423,210,0.415020


In [8]:
# 표기 불일치 확인
for side, name in [('left_only', '등록대수에만 있음'), ('right_only', '충전기에만 있음')]:
    rows = panel_outer.query(f'_merge == "{side}"')[['sido_short', 'sigungu']]
    print(f'\n[{name}] {len(rows)}건')
    print(rows.to_string(index=False))


[등록대수에만 있음] 0건
Empty DataFrame
Columns: [sido_short, sigungu]
Index: []

[충전기에만 있음] 16건
sido_short sigungu
        경기     고양시
        경기     성남시
        경기     안산시
        경기     안양시
        경기 안양시 만얀구
        경기     용인시
        경기 용인시 서인구
        경남     창원시
        경북     포항시
        전남  영암군삼호읍
        전북     전주시
        제주     한경면
        충남     천안시
        충북     청주시
        충북 청주시 픙덕구
        충북 청주시 홍덕구


# 등록대수 대비 충전기 과부족 지표 생성 (로그로그회귀)

In [9]:
# log-log 회귀로 "이 정도 등록대수면 보통 몇 대"인지 기대치를 구하고 실제값과의 괴리율(%)을 쓴다.
sigungu_panel['log_reg'] = np.log(sigungu_panel['registration'])
log_shape(sigungu_panel, 'log_reg 추가')


def loglog_expected_and_gap(df, y_col):
    """log(y) ~ log(registration) 단순회귀 -> (기대 충전기수, 기대치 대비 괴리율 %)."""
    y = np.log(df[y_col].clip(lower=0.5)) # 0대인 시군구의 log(0) = -inf 방지
    x = df['log_reg']
    X = np.column_stack([np.ones(len(x)), x]) # 절편항 + 기울기항
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    expected = np.exp(X @ beta) # 로그공간 예측 → 원 스케일 복원
    gap_pct = (df[y_col] / expected - 1) * 100 # +면 기대보다 많음, -면 부족
    return expected, gap_pct


for col, tag in [('total_chargers', 'total'),
                 ('fast_chargers', 'fast'),
                 ('slow_chargers', 'slow')]:
    sigungu_panel[f'expected_{tag}'], sigungu_panel[f'gap_pct_{tag}'] = \
        loglog_expected_and_gap(sigungu_panel, col)
log_shape(sigungu_panel, '기대치·괴리율 6컬럼')

# 괴리율이 규모와 무관해야 지표로서 의미가 있다 (0에 가까울수록 좋음)
r = sigungu_panel['gap_pct_total'].corr(sigungu_panel['log_reg'])
print(f'log(등록대수) vs gap_pct_total 상관계수: {r:.4f}')

print('\n===== 충전기 부족 하위 20 =====')
sigungu_panel.nsmallest(20, 'gap_pct_total')[
    ['sido_short', 'sigungu', 'registration', 'total_chargers', 'gap_pct_total']
].round(1)

[log_reg 추가                ]      250 행 ×   9 열  (행 +0 / 열 +1  ← 최종 패널)
[기대치·괴리율 6컬럼               ]      250 행 ×  15 열  (행 +0 / 열 +6  ← log_reg 추가)
log(등록대수) vs gap_pct_total 상관계수: 0.0286

===== 충전기 부족 하위 20 =====


,sido_short,sigungu,registration,total_chargers,gap_pct_total
201,전남,신안군,7628.0,178,-93.9
147,부산,중구,2772.0,71,-92.8
181,인천,계양구,14975.0,865,-85.6
87,경남,함안군,2575.0,176,-80.7
137,부산,동구,3604.0,311,-76.2
105,경북,울릉군,637.0,58,-71.5
180,인천,강화군,2733.0,336,-65.4
230,제주,제주시,40038.0,7176,-58.4
145,부산,연제구,6246.0,993,-57.8
210,전남,함평군,742.0,108,-55.0


# 가동률, 혼잡도 (+ 표준 데이터)

In [10]:
# 충전소 표준데이터: 충전기 상태코드 → 시군구별 가동률 / 혼잡도
# STAT 코드: 1 통신이상 / 2 충전대기 / 3 충전중 / 4 운영중지 / 5 점검중 / 9 미확인
# → 2,3 = 정상 응답 / 1,4,5 = 비정상 / 9 = 판단 불가

STD_COLS = ['STATID', 'CHGERID', 'STAT', 'ADDR', 'STATUPDDT', 'YEAR', 'BUSINM']
STAT_OK, STAT_BROKEN = [2, 3], [1, 4, 5]

charger_std = pd.read_csv(CHARGER_STD_FILE, usecols=STD_COLS)
log_shape(charger_std, '표준데이터 원본')

print(charger_std['STAT'].value_counts().sort_index()) # 코드 분포 확인

charger_std = attach_region_keys(charger_std, 'ADDR', label='표준데이터 주소 파싱')

charger_std['is_ok']     = charger_std['STAT'].isin(STAT_OK).astype(int)
charger_std['is_broken'] = charger_std['STAT'].isin(STAT_BROKEN).astype(int)
charger_std['is_busy']   = (charger_std['STAT'] == 3).astype(int) # 충전중 = 포화 신호
log_shape(charger_std, '상태 플래그 3컬럼')

status_by_sigungu = charger_std.groupby(['sido_short', 'sigungu']).agg(
    n_reported =('STATID', 'count'),
    n_ok       =('is_ok', 'sum'),
    n_broken   =('is_broken', 'sum'),
    n_busy     =('is_busy', 'sum'),
).reset_index()
status_by_sigungu['ok_rate']     = status_by_sigungu['n_ok'] / status_by_sigungu['n_reported'] * 100
status_by_sigungu['broken_rate'] = status_by_sigungu['n_broken'] / status_by_sigungu['n_reported'] * 100
status_by_sigungu['busy_rate']   = status_by_sigungu['n_busy'] / status_by_sigungu['n_ok'] * 100
log_shape(status_by_sigungu, '상태 시군구 집계')

MIN_REPORTED = 10 # 표본 10대 미만은 비율이 불안정 → 제외
status_stable = status_by_sigungu[status_by_sigungu['n_reported'] >= MIN_REPORTED].copy()
log_shape(status_stable, f'표본 {MIN_REPORTED}대 이상')

print(f'\n전국 평균 가동률: {status_stable["ok_rate"].mean():.1f}%')
print(f'전국 평균 혼잡도: {status_stable["busy_rate"].mean():.1f}%')

print('\n===== 가동률 하위 10 =====')
status_stable.nsmallest(10, 'ok_rate')[
    ['sido_short', 'sigungu', 'n_reported', 'ok_rate', 'broken_rate']
].round(1)

[표준데이터 원본                  ]  525,894 행 ×   7 열  (행 +525,644 / 열 -8  ← 기대치·괴리율 6컬럼)
STAT
1     11573
2    429310
3     60704
4       760
5      2118
9     21429
Name: count, dtype: int64
[표준데이터 주소 파싱 (실패 61,170건 제외)]  464,724 행 ×  10 열  (행 -61,170 / 열 +3  ← 표준데이터 원본)
[상태 플래그 3컬럼                ]  464,724 행 ×  13 열  (행 +0 / 열 +3  ← 표준데이터 주소 파싱 (실패 61,170건 제외))
[상태 시군구 집계                 ]      249 행 ×   9 열  (행 -464,475 / 열 -4  ← 상태 플래그 3컬럼)
[표본 10대 이상                 ]      229 행 ×   9 열  (행 -20 / 열 +0  ← 상태 시군구 집계)

전국 평균 가동률: 91.2%
전국 평균 혼잡도: 12.0%

===== 가동률 하위 10 =====


,sido_short,sigungu,n_reported,ok_rate,broken_rate
74,경남,산청군,435,34.3,63.0
207,전북,진안군,195,63.6,33.3
109,경북,청송군,197,66.0,11.7
105,경북,울릉군,36,66.7,11.1
180,인천,남구,28,67.9,0.0
16,강원,화천군,128,71.9,0.8
86,경남,하동군,359,74.9,14.2
68,경남,거창군,302,77.5,21.5
18,경기,가평군,699,77.5,5.3
195,전북,남원시,678,80.1,12.5


In [11]:
# 급속충전기 평균 용량
# '급속충전량'이 '100kW' 형태 문자열 → 숫자만 추출. 결측 많아 보조지표로만 사용

charger_ops['kw'] = pd.to_numeric(
    charger_ops['급속충전량'].astype(str).str.extract(r'(\d+)\s*kW')[0],
    errors='coerce'
)
log_shape(charger_ops, 'kw 컬럼 추가')

fast_with_kw = charger_ops[(charger_ops['is_fast'] == 1) & charger_ops['kw'].notna()]
n_fast = int(charger_ops['is_fast'].sum())
log_shape(fast_with_kw, '급속 & kW 보유')
print(f'급속 {n_fast:,}개 중 kW 정보 {len(fast_with_kw):,}개 ({len(fast_with_kw)/n_fast*100:.1f}%)')

kw_by_sigungu = fast_with_kw.groupby(['sido_short', 'sigungu']).agg(
    n_kw_reported=('kw', 'count'),
    avg_kw       =('kw', 'mean'),
).reset_index()
log_shape(kw_by_sigungu, 'kW 시군구 집계')
kw_by_sigungu.head()

[kw 컬럼 추가                  ]  272,771 행 ×  22 열  (행 +272,542 / 열 +13  ← 표본 10대 이상)
[급속 & kW 보유                ]    7,782 행 ×  22 열  (행 -264,989 / 열 +0  ← kw 컬럼 추가)
급속 31,003개 중 kW 정보 7,782개 (25.1%)
[kW 시군구 집계                 ]      256 행 ×   4 열  (행 -7,526 / 열 -18  ← 급속 & kW 보유)


,sido_short,sigungu,n_kw_reported,avg_kw
0,강원,강릉시,50,92.000000
1,강원,고성군,25,150.000000
2,강원,동해시,24,154.166667
3,강원,삼척시,57,114.035088
4,강원,속초시,28,137.500000


In [12]:
# 모든 지표를 시군구 패널에 병합
# left join인 이유: 상태·용량 지표에 결측이 있어도 패널 행을 잃지 않아야 함

sigungu_full = (
    sigungu_panel
    .merge(status_stable[['sido_short', 'sigungu', 'n_reported',
                          'ok_rate', 'broken_rate', 'busy_rate']],
           on=['sido_short', 'sigungu'], how='left')
    .merge(kw_by_sigungu, on=['sido_short', 'sigungu'], how='left')
)
log_shape(sigungu_full, '지표 병합 완료')

# 결측이 많은 지표는 아래 필터에서 지역이 통째로 빠지는 원인이 됨
print('\n지표별 결측 수:')
print(sigungu_full[['ok_rate', 'busy_rate', 'avg_kw']].isna().sum())
sigungu_full.head()

[지표 병합 완료                  ]      250 행 ×  21 열  (행 -6 / 열 +17  ← kW 시군구 집계)

지표별 결측 수:
ok_rate      30
busy_rate    30
avg_kw        2
dtype: int64


,sido_short,sigungu,registration,total_chargers,fast_chargers,slow_chargers,apt_charger_count,apt_charger_ratio,log_reg,expected_total,...,expected_fast,gap_pct_fast,expected_slow,gap_pct_slow,n_reported,ok_rate,broken_rate,busy_rate,n_kw_reported,avg_kw
0,강원,강릉시,3671.0,1473,273,1200,634,0.430414,8.208219,1332.251980,...,141.385878,93.088591,1172.128161,2.377883,2479.0,92.335619,3.025413,12.101354,50.0,92.000000
1,강원,고성군,364.0,228,62,166,44,0.192982,5.897154,111.905901,...,37.369537,65.910538,70.593651,135.148627,339.0,86.725664,2.949853,13.945578,25.0,150.000000
2,강원,동해시,872.0,476,56,420,286,0.600840,6.770789,285.433393,...,61.797756,-9.381822,204.189163,105.691621,851.0,94.359577,1.880141,9.090909,24.0,154.166667
3,강원,삼척시,729.0,507,142,365,154,0.303748,6.591674,235.576338,...,55.742229,154.744030,164.233660,122.244331,825.0,89.696970,5.212121,9.054054,57.0,114.035088
4,강원,속초시,887.0,506,83,423,210,0.415020,6.787845,290.699071,...,62.407602,32.996619,208.467222,102.909597,1120.0,94.375000,3.035714,14.096500,28.0,137.500000


# 3중고 분석 (세 테이블 모두 사용)

In [13]:
GAP_QUANTILE = 0.20
gap_cut     = sigungu_full['gap_pct_total'].quantile(GAP_QUANTILE)
ok_median   = sigungu_full['ok_rate'].median()
busy_median = sigungu_full['busy_rate'].median()
print(f'기준선 - 부족 컷 {gap_cut:.1f}% / 가동률 중앙값 {ok_median:.1f}% / 혼잡 중앙값 {busy_median:.1f}%')

triple_burden = sigungu_full[
    (sigungu_full['gap_pct_total'] <= gap_cut) &   # 충전기 수 하위 20%
    (sigungu_full['ok_rate']       <= ok_median) & # 가동률 중앙값 이하
    (sigungu_full['busy_rate']     >= busy_median) # 혼잡도 중앙값 이상
].copy()
log_shape(triple_burden, '3중고 지역 필터')

print('\n===== 3중고 지역 =====')
triple_burden[['sido_short', 'sigungu', 'registration', 'gap_pct_total',
               'ok_rate', 'busy_rate']].round(1).sort_values('gap_pct_total')

기준선 - 부족 컷 -25.4% / 가동률 중앙값 92.6% / 혼잡 중앙값 11.6%
[3중고 지역 필터                 ]        9 행 ×  21 열  (행 -241 / 열 +0  ← 지표 병합 완료)

===== 3중고 지역 =====


,sido_short,sigungu,registration,gap_pct_total,ok_rate,busy_rate
138,부산,중구,2772.0,-92.8,91.7,18.2
79,경남,함안군,2575.0,-80.7,92.0,12.7
97,경북,울릉군,637.0,-71.5,66.7,29.2
219,제주,제주시,40038.0,-58.4,87.1,15.3
69,경남,의령군,376.0,-50.8,91.4,13.5
238,충북,보은군,661.0,-46.7,89.3,15.9
173,인천,남동구,8627.0,-44.5,91.3,11.9
67,경남,산청군,719.0,-43.6,34.3,14.1
81,경남,합천군,866.0,-25.5,89.2,14.3


# 예측 모델

In [14]:
# 결핍지역 예측모델용 설명변수 준비 - 주택/인구
# 병합 기준은 sigungu_panel (가동률/혼잡도는 결측이 많아 모델에서 제외)

HOUSING_COLS = {'아파트비율': 'apt_ratio', '노후비율': 'old_housing_ratio',
                '자가주차장확보율': 'parking_ratio'}

housing = pd.read_csv(HOUSING_FILE, encoding='utf-8-sig')
log_shape(housing, '주택지표 원본')

pop = pd.read_csv(POP_FILE, encoding='utf-8-sig')
log_shape(pop, '인구 원본')

model_base = (
    sigungu_panel
    .merge(housing[['sido_short', 'sigungu'] + list(HOUSING_COLS)],
           on=['sido_short', 'sigungu'], how='left')
    .rename(columns=HOUSING_COLS)
    .merge(pop[['sido_short', 'sigungu', '인구수']].rename(columns={'인구수': 'population'}),
           on=['sido_short', 'sigungu'], how='left')
)
model_base['log_pop'] = np.log(model_base['population'])
log_shape(model_base, '주택·인구 병합')

print('\n병합 후 결측:')
print(model_base[['apt_ratio', 'old_housing_ratio', 'parking_ratio', 'population']].isna().sum())
model_base.head()

[주택지표 원본                   ]      252 행 ×   8 열  (행 +243 / 열 -13  ← 3중고 지역 필터)
[인구 원본                     ]      249 행 ×  29 열  (행 -3 / 열 +21  ← 주택지표 원본)
[주택·인구 병합                  ]      250 행 ×  20 열  (행 +1 / 열 -9  ← 인구 원본)

병합 후 결측:
apt_ratio            1
old_housing_ratio    1
parking_ratio        2
population           1
dtype: int64


,sido_short,sigungu,registration,total_chargers,fast_chargers,slow_chargers,apt_charger_count,apt_charger_ratio,log_reg,expected_total,gap_pct_total,expected_fast,gap_pct_fast,expected_slow,gap_pct_slow,apt_ratio,old_housing_ratio,parking_ratio,population,log_pop
0,강원,강릉시,3671.0,1473,273,1200,634,0.430414,8.208219,1332.251980,10.564670,141.385878,93.088591,1172.128161,2.377883,60.7,33.914667,83.1,212352.0,12.266001
1,강원,고성군,364.0,228,62,166,44,0.192982,5.897154,111.905901,103.742607,37.369537,65.910538,70.593651,135.148627,22.6,38.606864,81.3,27774.0,10.231856
2,강원,동해시,872.0,476,56,420,286,0.600840,6.770789,285.433393,66.763949,61.797756,-9.381822,204.189163,105.691621,68.2,35.193345,85.8,85673.0,11.358293
3,강원,삼척시,729.0,507,142,365,154,0.303748,6.591674,235.576338,115.216861,55.742229,154.744030,164.233660,122.244331,48.8,39.627096,84.1,62953.0,11.050144
4,강원,속초시,887.0,506,83,423,210,0.415020,6.787845,290.699071,74.063164,62.407602,32.996619,208.467222,102.909597,73.0,30.432725,87.4,79758.0,11.286752


In [15]:
# 결핍지역 예측모델용 설명변수 준비 - 보조금
# 보조금 데이터는 시군구가 아니라 시도/시 단위로 집계됨
# 광역시 → 시도 단위 (부산 해운대구 → 부산광역시)
# 다구시 → 시 단위 (경기 성남시 분당구 → 성남시)
# 나머지 → 시군구 그대로

SUBSIDY_COLS = {'접수율(%)': 'subsidy_receipt_rate', '예산소진율(%)': 'budget_exec_rate'}

# SIDO_FULL_TO_SHORT를 뒤집어 광역시만 추출
METRO_SHORT_TO_FULL = {short: full for full, short in SIDO_FULL_TO_SHORT.items()
                       if full.endswith(('특별시', '광역시'))}

subsidy = pd.read_excel(SUBSIDY_FILE, sheet_name='요약')[
    ['시도', '지역구분'] + list(SUBSIDY_COLS)
].rename(columns=SUBSIDY_COLS)
log_shape(subsidy, '보조금 원본')


def subsidy_lookup_key(row):
    """(sido_short, sigungu) -> 보조금 표의 (시도, 지역구분) 키."""
    sido, sgg = row['sido_short'], row['sigungu']
    if sido in METRO_SHORT_TO_FULL:
        return sido, METRO_SHORT_TO_FULL[sido]
    for city in CITIES_WITH_GU:
        if sgg.startswith(city):
            return sido, city
    return sido, sgg


keys = model_base.apply(subsidy_lookup_key, axis=1)
model_base['lk_sido']   = keys.apply(lambda t: t[0])
model_base['lk_region'] = keys.apply(lambda t: t[1])
log_shape(model_base, '보조금 조인키 추가')

model_base = model_base.merge(subsidy, left_on=['lk_sido', 'lk_region'],
                              right_on=['시도', '지역구분'], how='left')
log_shape(model_base, '보조금 병합')

print(f"\n보조금 미매칭: {model_base['subsidy_receipt_rate'].isna().sum()}건")
model_base.head()

[보조금 원본                    ]      161 행 ×   4 열  (행 -89 / 열 -16  ← 주택·인구 병합)
[보조금 조인키 추가                ]      250 행 ×  22 열  (행 +89 / 열 +18  ← 보조금 원본)
[보조금 병합                    ]      250 행 ×  26 열  (행 +0 / 열 +4  ← 보조금 조인키 추가)

보조금 미매칭: 2건


,sido_short,sigungu,registration,total_chargers,fast_chargers,slow_chargers,apt_charger_count,apt_charger_ratio,log_reg,expected_total,...,old_housing_ratio,parking_ratio,population,log_pop,lk_sido,lk_region,시도,지역구분,subsidy_receipt_rate,budget_exec_rate
0,강원,강릉시,3671.0,1473,273,1200,634,0.430414,8.208219,1332.251980,...,33.914667,83.1,212352.0,12.266001,강원,강릉시,강원,강릉시,49.0,49.0
1,강원,고성군,364.0,228,62,166,44,0.192982,5.897154,111.905901,...,38.606864,81.3,27774.0,10.231856,강원,고성군,강원,고성군,102.0,102.0
2,강원,동해시,872.0,476,56,420,286,0.600840,6.770789,285.433393,...,35.193345,85.8,85673.0,11.358293,강원,동해시,강원,동해시,49.0,49.0
3,강원,삼척시,729.0,507,142,365,154,0.303748,6.591674,235.576338,...,39.627096,84.1,62953.0,11.050144,강원,삼척시,강원,삼척시,77.0,77.0
4,강원,속초시,887.0,506,83,423,210,0.415020,6.787845,290.699071,...,30.432725,87.4,79758.0,11.286752,강원,속초시,강원,속초시,286.0,286.0


In [16]:
# 라벨: gap_pct_total 하위 20% = 결핍
# 단순 (충전기수/등록대수) 비율로 라벨을 만들면 소규모 지역이 과도하게 결핍으로 잡히는
# 순환성 문제가 있어 규모효과를 제거한 로그로그 잔차 기준으로 정의

DEFICIT_QUANTILE = 0.20
FEATURES = ['log_reg', 'log_pop', 'apt_ratio', 'old_housing_ratio',
            'parking_ratio', 'subsidy_receipt_rate', 'budget_exec_rate']

cutoff = model_base['gap_pct_total'].quantile(DEFICIT_QUANTILE)
model_base['is_deficit'] = (model_base['gap_pct_total'] <= cutoff).astype(int)
log_shape(model_base, 'is_deficit 라벨 추가')

model_df = model_base.dropna(subset=FEATURES).copy() # 설명변수 결측 행 제거
log_shape(model_df, '결측 제거 (모델 입력)')

X = model_df[FEATURES].values
y = model_df['is_deficit'].values
prevalence = y.mean()
print(f'\n컷오프 {cutoff:.1f}% | N = {len(y)}, 결핍비율 = {prevalence:.3f} (양성 {y.sum()}개)')

[is_deficit 라벨 추가          ]      250 행 ×  27 열  (행 +0 / 열 +1  ← 보조금 병합)
[결측 제거 (모델 입력)             ]      246 행 ×  27 열  (행 -4 / 열 +0  ← is_deficit 라벨 추가)

컷오프 -25.4% | N = 246, 결핍비율 = 0.199 (양성 49개)


In [17]:
# 5-fold pooled out-of-fold 확률 → Youden's J로 단일 threshold 결정
# 표본이 작아 단일 train/test split은 노이즈가 커서 전체를 한 번씩 test로 사용

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
THRESHOLDS = {} 

def cv_evaluate(name, make_clf):
    proba = np.zeros(len(y))
    for train_idx, test_idx in SKF.split(X, y):
        clf = make_clf()
        clf.fit(X[train_idx], y[train_idx])
        proba[test_idx] = clf.predict_proba(X[test_idx])[:, 1]

    fpr, tpr, thresholds = roc_curve(y, proba)
    best_thresh = thresholds[np.argmax(tpr - fpr)] # Youden's J
    THRESHOLDS[name] = float(best_thresh)

    pred = (proba >= best_thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {
        '모델': name,
        'AUROC': roc_auc_score(y, proba),
        'BalancedAccuracy': balanced_accuracy_score(y, pred),
        'Accuracy': accuracy_score(y, pred),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        'Sensitivity': tp / (tp + fn) if (tp + fn) > 0 else np.nan,
        'AUPRC': average_precision_score(y, proba),
        'F1': f1_score(y, pred, zero_division=0),
        'Precision': precision_score(y, pred, zero_division=0),
    }

In [18]:
# 동일 데이터 · 동일 평가방식으로 10종 비교
POS_WEIGHT = (1 - prevalence) / prevalence # class_weight 미지원 모델용 불균형 보정

MODELS = {
    'DecisionTree': lambda: DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=5, random_state=42, class_weight='balanced'),
    'XGBoost': lambda: XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42,
        scale_pos_weight=POS_WEIGHT, eval_metric='logloss'),
    'RandomForest': lambda: RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_leaf=5, random_state=42, class_weight='balanced'),
    'GBM': lambda: GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42),
    'MLP': lambda: Pipeline([('scaler', StandardScaler()), # fold별 스케일링으로 누수 방지
        ('clf', MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=2000,
                              random_state=42, early_stopping=True))]),
    'Logistic regression': lambda: Pipeline([('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))]),
    'Extra trees': lambda: ExtraTreesClassifier(
        n_estimators=300, max_depth=5, min_samples_leaf=5, random_state=42, class_weight='balanced'),
    'LightGBM': lambda: LGBMClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42,
        class_weight='balanced', verbose=-1),
    'AdaBoost': lambda: AdaBoostClassifier(
        n_estimators=150, learning_rate=0.05, random_state=42),
    'CatBoost': lambda: CatBoostClassifier(
        iterations=200, depth=3, learning_rate=0.05, random_state=42,
        scale_pos_weight=POS_WEIGHT, verbose=False),
}

single_df = (
    pd.DataFrame([cv_evaluate(name, fn) for name, fn in MODELS.items()])
      .set_index('모델').round(3).sort_values('AUROC', ascending=False)
)
log_shape(single_df, '단일모델 결과')
single_df

[단일모델 결과                   ]       10 행 ×   8 열  (행 -236 / 열 -19  ← 결측 제거 (모델 입력))


,AUROC,BalancedAccuracy,Accuracy,Specificity,Sensitivity,AUPRC,F1,Precision
모델,,,,,,,,
Logistic regression,0.867,0.809,0.768,0.741,0.878,0.640,0.601,0.457
XGBoost,0.865,0.807,0.728,0.675,0.939,0.608,0.579,0.418
LightGBM,0.861,0.794,0.756,0.731,0.857,0.648,0.583,0.442
CatBoost,0.853,0.812,0.821,0.827,0.796,0.613,0.639,0.534
RandomForest,0.839,0.791,0.789,0.787,0.796,0.552,0.600,0.481
GBM,0.839,0.802,0.768,0.746,0.857,0.606,0.596,0.457
Extra trees,0.807,0.761,0.764,0.766,0.755,0.446,0.561,0.446
AdaBoost,0.773,0.715,0.642,0.594,0.837,0.455,0.482,0.339
DecisionTree,0.756,0.738,0.789,0.822,0.653,0.410,0.552,0.478


In [19]:
# AUROC 상위 모델을 Soft Voting / Weighted Soft Voting / Stacking으로 조합

individual_auroc = single_df['AUROC'].to_dict()

# 앙상블 이름에 넣을 약칭
MODEL_ABBR = {
    'DecisionTree': 'DT', 'XGBoost': 'XGB', 'RandomForest': 'RF', 'GBM': 'GBM',
    'MLP': 'MLP', 'Logistic regression': 'LR', 'Extra trees': 'ET',
    'LightGBM': 'LGBM', 'AdaBoost': 'ADA', 'CatBoost': 'CAT',
}

def build_ensembles(model_names):
    est = lambda: [(n, MODELS[n]()) for n in model_names] # fold마다 새 인스턴스
    w = np.array([individual_auroc[n] for n in model_names])
    w = w / w.sum() # AUROC 비례 가중치
    return {
        'Soft Voting':          lambda: VotingClassifier(estimators=est(), voting='soft'),
        'Weighted Soft Voting': lambda: VotingClassifier(estimators=est(), voting='soft', weights=w),
        # Stacking 내부 cv=5는 메타모델용 OOF 피처 생성용
        'Stacking':             lambda: StackingClassifier(
            estimators=est(), final_estimator=LogisticRegression(max_iter=1000),
            cv=5, stack_method='predict_proba'),
    }

top_models = single_df.index.tolist()
ensemble_results = []
for k in (2, 3, 4):
    members = top_models[:k]
    combo = '+'.join(MODEL_ABBR.get(n, n) for n in members) 
    for name, make_clf in build_ensembles(members).items():
        ensemble_results.append(cv_evaluate(f'{name} ({combo})', make_clf))

ensemble_df = pd.DataFrame(ensemble_results).set_index('모델').round(3)
log_shape(ensemble_df, '앙상블 결과')

# 앙상블 구성 참고표
print('===== 앙상블 조합 =====')
for k in (2, 3, 4):
    print(f'상위{k}: ' + ' + '.join(top_models[:k]))

combined = pd.concat([single_df, ensemble_df]).sort_values('AUROC', ascending=False)
log_shape(combined, '전체 통합 순위')
combined

[앙상블 결과                    ]        9 행 ×   8 열  (행 -1 / 열 +0  ← 단일모델 결과)
===== 앙상블 조합 =====
상위2: Logistic regression + XGBoost
상위3: Logistic regression + XGBoost + LightGBM
상위4: Logistic regression + XGBoost + LightGBM + CatBoost
[전체 통합 순위                  ]       19 행 ×   8 열  (행 +10 / 열 +0  ← 앙상블 결과)


,AUROC,BalancedAccuracy,Accuracy,Specificity,Sensitivity,AUPRC,F1,Precision
모델,,,,,,,,
Weighted Soft Voting (LR+XGB+LGBM),0.888,0.835,0.797,0.772,0.898,0.659,0.638,0.494
Soft Voting (LR+XGB+LGBM),0.888,0.835,0.797,0.772,0.898,0.659,0.638,0.494
Soft Voting (LR+XGB),0.885,0.835,0.785,0.751,0.918,0.653,0.629,0.479
Weighted Soft Voting (LR+XGB),0.885,0.835,0.785,0.751,0.918,0.653,0.629,0.479
Stacking (LR+XGB+LGBM),0.883,0.840,0.805,0.782,0.898,0.655,0.647,0.506
Stacking (LR+XGB+LGBM+CAT),0.881,0.845,0.801,0.772,0.918,0.655,0.647,0.500
Weighted Soft Voting (LR+XGB+LGBM+CAT),0.881,0.842,0.797,0.766,0.918,0.645,0.643,0.495
Soft Voting (LR+XGB+LGBM+CAT),0.881,0.842,0.797,0.766,0.918,0.646,0.643,0.495
Stacking (LR+XGB),0.881,0.827,0.785,0.756,0.898,0.650,0.624,0.478


In [20]:
shape_history = pd.DataFrame(_shape_log, columns=['step', 'rows', 'cols'])
shape_history.insert(0, 'order', range(1, len(shape_history) + 1))

shape_history.to_csv(DATA_DIR / 'shape_history.csv', index=False, encoding='utf-8-sig')
print(f'저장 완료: {DATA_DIR / "shape_history.csv"}\n')

shape_history

저장 완료: /Users/ichaeu/Desktop/EVen/shape_history.csv



,order,step,rows,cols
0,1,등록대수 원본,508,8
1,2,연료=전기 필터,508,8
2,3,지역 키 컬럼 추가,508,10
3,4,등록대수 시군구 집계,250,3
4,5,충전소 원본,273281,15
5,6,충전소 주소 파싱 (실패 510건 제외),272771,18
6,7,플래그 컬럼 추가,272771,21
7,8,충전기 시군구 집계,266,6
8,9,비율 파생변수 추가,266,7
9,10,outer merge,266,9


In [21]:
# 배포용 모델은 전체 데이터로 다시 학습
# threshold는 CV에서 구한 값을 그대로 사용

ALL_MODELS = {**MODELS} # 단일모델
for k in (2, 3, 4): # 앙상블도 후보에 포함
    members = top_models[:k]
    combo = '+'.join(MODEL_ABBR.get(n, n) for n in members)
    for name, make_clf in build_ensembles(members).items():
        ALL_MODELS[f'{name} ({combo})'] = make_clf

BEST_NAME = combined.index[0] # AUROC 1위
best_model = ALL_MODELS[BEST_NAME]()
best_model.fit(X, y)
best_thresh = THRESHOLDS[BEST_NAME]

print(f'최종 모델: {BEST_NAME}')
print(f'AUROC {combined.loc[BEST_NAME, "AUROC"]:.3f} | threshold {best_thresh:.4f}')

# 성능 테이블 저장
metrics_path = DATA_DIR / 'model_performance.xlsx'
with pd.ExcelWriter(metrics_path, engine='openpyxl') as writer:
    combined.to_excel(writer, sheet_name='전체순위')
    single_df.to_excel(writer, sheet_name='단일모델')
    ensemble_df.to_excel(writer, sheet_name='앙상블')
    shape_history.to_excel(writer, sheet_name='전처리이력', index=False)
    pd.DataFrame({
        '항목': ['최종모델', 'threshold', 'N', '양성수', '결핍비율',
                 '결핍컷오프(gap_pct)', '변수'],
        '값': [BEST_NAME, round(best_thresh, 4), len(y), int(y.sum()),
               round(prevalence, 3), round(cutoff, 2), ', '.join(FEATURES)],
    }).to_excel(writer, sheet_name='요약', index=False)
print(f'성능 테이블 저장: {metrics_path}')

# 모델 저장
model_path = DATA_DIR / 'deficit_model.pkl'
joblib.dump({
    'model': best_model,
    'model_name': BEST_NAME,
    'threshold': best_thresh,
    'features': FEATURES,
    'deficit_quantile': DEFICIT_QUANTILE,
    'gap_pct_cutoff': float(cutoff),
    'prevalence': float(prevalence),
    'cv_metrics': combined.loc[BEST_NAME].to_dict(),
}, model_path)
print(f'모델 저장: {model_path}')

최종 모델: Weighted Soft Voting (LR+XGB+LGBM)
AUROC 0.888 | threshold 0.3086
성능 테이블 저장: /Users/ichaeu/Desktop/EVen/model_performance.xlsx
모델 저장: /Users/ichaeu/Desktop/EVen/deficit_model.pkl


# 분석 결과 Streamlit 연동용 코드

In [ ]:
# 분석별 결과 테이블을 각각 xlsx로 저장 (컬럼명 한글화)
RESULTS_DIR = DATA_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

# 엑셀 출력용 컬럼명 매핑 (내부 데이터프레임은 영문 유지)
COL_KR = {
    'sido_short': '시도', 'sigungu': '시군구',
    'registration': '전기차 등록대수',
    'total_chargers': '충전기 수', 'fast_chargers': '급속', 'slow_chargers': '완속',
    'apt_charger_count': '공동주택 설치', 'apt_charger_ratio': '공동주택 비중',
    'expected_total': '기대 충전기 수',
    'expected_fast': '기대 급속', 'expected_slow': '기대 완속',
    'gap_pct_total': '괴리율(%)', 'gap_pct_fast': '급속 괴리율(%)',
    'gap_pct_slow': '완속 괴리율(%)',
    'log_reg': 'log(등록대수)',
    'n_reported': '상태보고 충전기', 'n_ok': '정상', 'n_broken': '고장', 'n_busy': '충전중',
    'ok_rate': '가동률(%)', 'broken_rate': '고장률(%)', 'busy_rate': '혼잡도(%)',
    'n_kw_reported': 'kW 표본수', 'avg_kw': '평균 출력(kW)',
}

def to_kr(df, cols=None):
    """엑셀 저장용: 컬럼 선택 + 한글명으로 변환."""
    out = df[cols] if cols else df
    return out.rename(columns=COL_KR)

# 1) 개수 갭
gap_cols = ['sido_short', 'sigungu', 'registration', 'total_chargers',
            'fast_chargers', 'slow_chargers', 'expected_total',
            'gap_pct_total', 'gap_pct_fast', 'gap_pct_slow']
gap_table = to_kr(sigungu_panel, gap_cols).sort_values('괴리율(%)')

with pd.ExcelWriter(RESULTS_DIR / '01_개수갭.xlsx', engine='openpyxl') as w:
    gap_table.round(2).to_excel(w, sheet_name='전체', index=False)
    gap_table.head(20).round(2).to_excel(w, sheet_name='부족 하위20', index=False)

# 2) 가동률·혼잡도
status_kr = to_kr(status_stable).round(2)
with pd.ExcelWriter(RESULTS_DIR / '02_가동률_혼잡도.xlsx', engine='openpyxl') as w:
    status_kr.to_excel(w, sheet_name='전체', index=False)
    status_kr.nsmallest(10, '가동률(%)').to_excel(w, sheet_name='가동률 하위10', index=False)
    status_kr.nlargest(10, '혼잡도(%)').to_excel(w, sheet_name='혼잡도 상위10', index=False)
    to_kr(status_by_sigungu).round(2).to_excel(
        w, sheet_name='표본10미만 포함', index=False)

# 3) 급속 용량
to_kr(kw_by_sigungu).round(2).to_excel(RESULTS_DIR / '03_급속용량.xlsx', index=False)

# 4) 3중고
triple_cols = ['sido_short', 'sigungu', 'registration', 'total_chargers',
               'gap_pct_total', 'ok_rate', 'busy_rate']
with pd.ExcelWriter(RESULTS_DIR / '04_3중고_지역.xlsx', engine='openpyxl') as w:
    to_kr(triple_burden, triple_cols).round(2).sort_values('괴리율(%)').to_excel(
        w, sheet_name='3중고 지역', index=False)
    pd.DataFrame({
        '기준': ['개수 부족 컷(하위20%)', '가동률 중앙값', '혼잡도 중앙값', '해당 지역 수'],
        '값': [round(gap_cut, 2), round(ok_median, 2), round(busy_median, 2),
               len(triple_burden)],
    }).to_excel(w, sheet_name='판정 기준', index=False)
    to_kr(sigungu_full).round(2).to_excel(w, sheet_name='통합지표 전체', index=False)

print(f'저장 완료: {RESULTS_DIR}')
for f in sorted(RESULTS_DIR.glob('*.xlsx')):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.0f} KB)')

저장 완료: /Users/ichaeu/Desktop/EVen/results
  01_개수갭.xlsx  (23 KB)
  02_가동률_혼잡도.xlsx  (33 KB)
  03_급속용량.xlsx  (12 KB)
  04_3중고지역.xlsx  (37 KB)
